# RAG の評価セットを用意する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/20_rag/rag_eval_dataset.ipynb)

NLP クックブック 20-08 ／ RAG (検索拡張生成)

RAG を評価するための「質問・正解・正解が書かれた文書」の組を用意したい。既存の QA データから作る方法と、自分の文書から LLM で作る方法を知りたい。

In [ ]:
!pip install -q datasets

In [ ]:
import json
from datasets import load_dataset

# 1. 既存の読解データセットから作る: JSQuAD の段落を検索対象の文書集合に、質問と答えを評価セットにする
ds = load_dataset("sbintuitions/JSQuAD", split="validation")
passages, pid, seen_q, eval_set = [], {}, set(), []
for ex in ds:
    context = ex["context"].split(" [SEP] ", 1)[-1]
    if context not in pid:
        pid[context] = len(passages)
        passages.append({"id": len(passages), "title": ex["title"], "text": context})
for ex in ds.shuffle(seed=0):
    context = ex["context"].split(" [SEP] ", 1)[-1]
    if pid[context] in seen_q:              # 1 つの段落から 1 問だけ使う
        continue
    seen_q.add(pid[context])
    eval_set.append({"question": ex["question"], "answers": sorted(set(ex["answers"]["text"])), "gold_passage": pid[context], "answerable": True})
    if len(eval_set) == 60:
        break
print(len(passages), "段落 /", len(eval_set), "問")
print(eval_set[0], "\n ", passages[eval_set[0]["gold_passage"]]["text"][:80])

In [ ]:
# 2. 文書集合に答えがない質問も入れる (「わかりません」と言えるかを測るため)
unanswerable = ["2030 年の冬季オリンピックの開催地はどこですか。", "私の昨日の夕食は何でしたか。", "この文書集合を作った人の名前は何ですか。",
                "来週の東京の天気はどうなりますか。", "円周率の小数第 100 位の数字は何ですか。", "現在の日本の首相の愛読書は何ですか。"]
eval_set += [{"question": q, "answers": [], "gold_passage": None, "answerable": False} for q in unanswerable]

with open("rag_passages.jsonl", "w", encoding="utf-8") as f:
    f.writelines(json.dumps(p, ensure_ascii=False) + "\n" for p in passages)
with open("rag_eval.jsonl", "w", encoding="utf-8") as f:
    f.writelines(json.dumps(e, ensure_ascii=False) + "\n" for e in eval_set)
print(len(eval_set), "問を保存 (うち答えなし", sum(not e["answerable"] for e in eval_set), "問)")

In [ ]:
# 3. 評価セットの性質を確かめる: 質問と正解段落の語の重なりが大きいと、キーワード検索に有利な「易しい」評価になる
import re
import numpy as np

chars = lambda s: set(re.findall(r"[一-龥ァ-ヶー]{2,}", s))
overlap = [len(chars(e["question"]) & chars(passages[e["gold_passage"]]["text"])) / max(1, len(chars(e["question"]))) for e in eval_set if e["answerable"]]
print(f"質問の語のうち正解段落にも出てくる割合: 平均 {np.mean(overlap):.2f}")
print("答えの長さ (文字):", np.percentile([len(e["answers"][0]) for e in eval_set if e["answerable"]], [25, 50, 75]))

- 評価セットに必要なのは 3 つ: 質問、正解 (言い方の違う複数の正解があるとよい)、正解が書かれた文書の ID。文書 ID があると、検索の失敗と生成の失敗を分けて調べられる。
- 既存の読解データセットの質問は、段落を見ながら作られているので、段落と同じ語を使いがち。実際のユーザーの質問より検索が易しい点に注意する。
- 自分の文書で作るなら、(1) 実際の問い合わせログから集める、(2) 人が文書を読んで質問を書く、(3) LLM に文書から質問を作らせて人が確認する (質問応答の章の質問生成を参照)、の順に質が高い。
- 答えのない質問を 1〜2 割入れる。入れないと「何を聞かれても答える」システムが高評価になってしまう。
- 50〜100 問でも、設定の良し悪しの比較には使える。差が小さいときは問数を増やすか、信頼区間を付ける。